# DarkTrace — Full Pipeline Runner (Kaggle)

Runs **every phase end-to-end** and produces all real results, figures, consolidated
tables, and the **statistical-validation** outputs (CIs + significance tests).

**Phases:** traffic · text · multilingual · multilingual cross-domain (Hindi/Arabic) ·
explainable scoring · evidence integrity (hash-chain) · STIX export · integration ablation ·
**statistics** (`exp_stats`).

## Recommended Kaggle settings
- **Accelerator:** `GPU T4 x2` (needed only for the multilingual transformer phase; everything else is CPU).
  Phase runs on CPU with a TF-IDF fallback if no GPU, but transformer numbers need the GPU.
- **Internet:** `On` (to clone the repo, pip-install pinned deps, and download the cross-domain corpora).
- **Persistence:** outputs are written under `/kaggle/working/DTPaper/results/`.

> Real-data-only: a phase whose dataset is missing **fails clearly** (exit code 2) rather than
> fabricating data. Cross-domain Hindi/Arabic is **optional** and is skipped (not failed) if absent.


## 1. Get the code

Option A clones the public repo. If your latest local changes aren't pushed, use Option B
(attach this checkout as a private Kaggle Dataset) instead.


In [ ]:
# Option A — clone (internet ON). Replace with your fork if needed.
!rm -rf /kaggle/working/DTPaper
!git clone https://github.com/coderbpl/DTPaper.git /kaggle/working/DTPaper
%cd /kaggle/working/DTPaper

# Option B — attach a private Kaggle Dataset containing the repo, then:
# !cp -R /kaggle/input/<your-dtpaper-dataset>/DTPaper /kaggle/working/DTPaper
# %cd /kaggle/working/DTPaper
!pwd && ls

## 2. Install pinned dependencies

The stats layer is deterministic on the pinned stack. Transformer phase needs `torch`/`transformers`
(preinstalled on Kaggle GPU images).


In [ ]:
# Core (pinned for reproducible stats). Kaggle usually has compatible versions already.
!python -m pip install -q -r requirements.txt || true
# Make sure the essentials are importable:
import importlib, sys
for m in ['numpy','pandas','scipy','sklearn']:
    print(m, '->', importlib.import_module(m).__version__)
# Optional (transformer phase): transformers, torch, stix2
for m in ['torch','transformers','stix2']:
    try: print(m, '->', importlib.import_module(m).__version__)
    except Exception as e: print(m, 'MISSING (', e, ')')

## 3. Stage the datasets into `data/raw/`

Attach your private Kaggle Datasets, then copy them to the expected paths. Respect the UNB CIC
and CoDA/DUTA terms — create **private** datasets; do not redistribute restricted data.

Expected: `data/raw/CIC-Darknet2020.csv` and `data/raw/coda.csv`.


In [ ]:
# Discover what's attached:
!find /kaggle/input -maxdepth 3 -type f \( -name '*.csv' -o -name '*.parquet' \) | sort | head -40

In [ ]:
from pathlib import Path
import shutil
raw = Path('data/raw'); raw.mkdir(parents=True, exist_ok=True)

# --- EDIT these source paths to match your attached datasets ---
CIC_SRC  = '/kaggle/input/<your-cic-dataset>/CIC-Darknet2020.csv'
CODA_SRC = '/kaggle/input/<your-coda-dataset>/coda.csv'

for src, dst in [(CIC_SRC, raw/'CIC-Darknet2020.csv'), (CODA_SRC, raw/'coda.csv')]:
    if Path(src).exists():
        shutil.copyfile(src, dst); print('staged', dst)
    else:
        print('!! NOT FOUND:', src, '— edit the path above')
print(sorted(p.name for p in raw.iterdir()))

### 3b. (Optional) Export CoDA from Hugging Face

Only if you have not prepared `coda.csv` yet and internet is ON. Skip if already staged above.


In [ ]:
# from pathlib import Path
# from datasets import load_dataset
# Path('data/raw').mkdir(parents=True, exist_ok=True)
# ds = load_dataset('s2w-ai/CoDA')
# split = ds['train'] if 'train' in ds else ds[list(ds.keys())[0]]
# df = split.to_pandas(); print(df.columns.tolist())
# # rename to text/label if needed, then:
# df.to_csv('data/raw/coda.csv', index=False)

### 3c. (Optional) Cross-domain Hindi/Arabic corpora

For Phase 2b (cross-domain proxy). Internet required. If it fails, the phase is **skipped**, not failed.


In [ ]:
!python download_corpora.py || echo 'cross-domain download failed -> Phase 2b will be SKIPPED'
!ls -la data/raw | sed -n '1,20p'

## 4. (Optional) Use the heavier multilingual transformer

Default is `distilbert-base-multilingual-cased`. Switch to `xlm-roberta-base` for the stronger
(GPU-heavier) model used in the manuscript discussion.


In [ ]:
import json
p = 'configs/multilingual.json'
cfg = json.load(open(p))
# cfg['model']['name'] = 'xlm-roberta-base'   # uncomment for XLM-R
# cfg['model']['epochs'] = 3
json.dump(cfg, open(p,'w'), indent=2)
print('multilingual model:', cfg['model']['name'])

## 5. (Optional) Smoke test

Verifies the code path only; results are non-reportable.


In [ ]:
!python -m src.exp_traffic --config configs/traffic.json --smoke-test || true
!python -m src.exp_text    --config configs/text.json    --smoke-test || true

## 6. Run the **full** pipeline (all phases + stats)

`run_all.py` runs every phase in dependency order, then figures, consolidated tables, and
`exp_stats` (CIs + significance tests). Each experiment also **saves per-instance predictions**
to `results/preds/`, which lets `exp_stats` compute the paired-bootstrap and per-language CIs.

To skip slow phases: `python run_all.py --skip exp_multilingual exp_ablation`.

**New phases now included automatically:** `exp_text_transformer` (XLM-R on CoDA text — **needs GPU**; skipped on CPU) and the full `exp_stats` battery, which now emits a **critical-difference diagram** (`figures/fig_cd_diagram.png`) once ≥3 text classifiers exist, and **per-language macro-F1 bootstrap CIs** from the saved multilingual predictions.


In [ ]:
!python run_all.py

## 7. Statistical validation (explicit)

`run_all` already ran this, but you can re-run it standalone anytime. It auto-detects `results/`.
With saved predictions present it adds the paired-bootstrap macro-F1 difference (scoring),
per-language macro-F1 bootstrap CIs, and Friedman+Nemenyi (when ≥3 comparable models exist).


In [ ]:
!python -m src.exp_stats --results-dir results
import json
s = json.load(open('results/tables/stats_results.json'))
print('sections:', list(s.keys()))
print('text McNemar:', s.get('classification_text',{}).get('mcnemar'))
print('full battery:', s.get('full_battery',{}).get('status'))

## 8. Inspect results


In [ ]:
!echo '===== ALL_RESULTS.md =====' && cat results/tables/ALL_RESULTS.md 2>/dev/null | head -80
!echo && echo '===== tables =====' && ls -lh results/tables
!echo && echo '===== figures =====' && ls -lh results/figures
!echo && echo '===== stats CSV =====' && cat results/tables/table_stats.csv 2>/dev/null

## 9. Package results for download

Zips `results/` (tables, figures, logs, preds, stix, ledger) into `/kaggle/working` so you can
download it and drop the numbers straight into the manuscript.


In [ ]:
import shutil, os
os.makedirs('/kaggle/working', exist_ok=True)
out = shutil.make_archive('/kaggle/working/darktrace_results', 'zip', 'results')
print('wrote', out, os.path.getsize(out), 'bytes')
# Optional: also copy a flat snapshot dir named darktrace_results/ for the paper repo layout
if os.path.isdir('results'):
    shutil.copytree('results', '/kaggle/working/darktrace_results', dirs_exist_ok=True)
    print('also copied results/ -> /kaggle/working/darktrace_results/')

---
### Mapping outputs to the manuscript
| File | Manuscript table/figure |
|---|---|
| `table6_text.csv`, `table6_traffic.csv` | Classification (Table 1 / Tab.~ref{tab:r-cls}) |
| `table7_multilingual.csv`, `table7b_crossdomain.csv` | Multilingual (Tab.~ref{tab:r-ml},~ref{tab:r-ml2}) |
| `table10_scoring.csv` | Severity scoring (Tab.~ref{tab:r-sev}) |
| `table8_blockchain.csv` | Evidence integrity (Tab.~ref{tab:r-int}) |
| `table9_stix.csv` | STIX export |
| `table11_ablation.csv` | Integration ablation (Tab.~ref{tab:r-abl}) |
| `stats_results.json`, `table_stats.csv` | Statistical validation (Sec. Statistical Validation) |
| `text_transformer_results.json`, `folds_Transformer.json` | Transformer-on-CoDA baseline + Friedman input |
| `figures/fig_cd_diagram.png` | Critical-difference diagram (Friedman+Nemenyi) |
| `preds/multilingual_*.npz` | Per-language macro-F1 bootstrap CIs |
| `figures/*.png` | All result figures |

Rebuild the merged PDF locally with `make manuscript` after dropping the new numbers in.
